# 🎯 AI Interview Agent - Google Colab Setup

Run this notebook to conduct real-time interviews using the AI Interview Agent!

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q fastapi uvicorn langchain langgraph langchain-openai langchain-community faiss-cpu sentence-transformers python-dotenv pypdf streamlit pyngrok requests

## Step 2: Set Your OpenRouter API Key

In [ ]:
import os
from google.colab import userdata

# Get API key from Colab Secrets
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
    os.environ['OPENROUTER_API_KEY'] = api_key
    print("✅ API key loaded from Colab Secrets")
except:
    print("⚠️ Could not load API key from Secrets.")
    print("Go to: Secrets (key icon) > Add new secret > OPENROUTER_API_KEY")
    api_key = input("Or paste your API key here: ")
    os.environ['OPENROUTER_API_KEY'] = api_key

## Step 3: Clone the Repository

In [ ]:
import os
import subprocess

# Clone repo (or use existing)
if not os.path.exists('AI-Interview-Agent'):
    !git clone https://github.com/yourusername/AI-Interview-Agent.git

os.chdir('AI-Interview-Agent')
print("✅ Ready to go!")

## Step 4: Option A - Use Streamlit Interface (Recommended)

In [ ]:
# Start API server
import subprocess
import time
from pyngrok import ngrok
import threading

# Start FastAPI server in background
def start_server():
    subprocess.run(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

time.sleep(3)
print("✅ API server started")

In [ ]:
# Get public URL with ngrok
from pyngrok import ngrok
import time

# Create ngrok tunnel
public_url = ngrok.connect(8000)
print(f"🌐 Public API URL: {public_url}")
print(f"📊 API Docs: {public_url}/docs")

# Save for later use
with open('/tmp/ngrok_url.txt', 'w') as f:
    f.write(str(public_url))

In [ ]:
# Run Streamlit app
!streamlit run app_streamlit.py --server.port 8501 --server.headless true

---

## Step 4: Option B - Direct Python Chat (Alternative)

In [ ]:
import requests
import json

# Read ngrok URL from file
try:
    with open('/tmp/ngrok_url.txt', 'r') as f:
        api_url = f.read().strip()
except:
    api_url = "http://localhost:8000"  # Local fallback

print(f"Using API: {api_url}")

# Interview settings
job_role = "Backend Developer"  # Change this
company_context = ""  # Optional

class InterviewChat:
    def __init__(self, api_url, job_role, context=""):
        self.api_url = api_url
        self.job_role = job_role
        self.context = context
        self.current_question = ""
        self.scores = []
        self.turn = 0
    
    def start(self):
        """Get first question"""
        try:
            resp = requests.post(
                f"{self.api_url}/interview",
                json={"job_role": self.job_role, "candidate_answer": ""},
                timeout=30
            )
            if resp.status_code == 200:
                data = resp.json()
                self.current_question = data.get('question', '')
                print(f"\n🎤 Interviewer: {self.current_question}\n")
            else:
                print(f"❌ Error: {resp.status_code}")
        except Exception as e:
            print(f"❌ Connection error: {e}")
    
    def answer(self, text):
        """Send answer and get feedback + follow-up"""
        self.turn += 1
        print(f"\nYou: {text}\n")
        
        try:
            resp = requests.post(
                f"{self.api_url}/interview/evaluate-with-followup",
                json={
                    "job_role": self.job_role,
                    "question": self.current_question,
                    "candidate_answer": text,
                    "company_context": self.context
                },
                timeout=30
            )
            if resp.status_code == 200:
                data = resp.json()
                
                # Show evaluation
                if 'evaluation' in data:
                    print(f"📋 Feedback:\n{data['evaluation']}\n")
                    
                    # Extract score
                    if 'Score:' in data['evaluation']:
                        try:
                            score = int(data['evaluation'].split('Score:')[1].split('/')[0])
                            self.scores.append(score)
                        except:
                            pass
                
                # Show next question
                if 'suggested_follow_up' in data:
                    self.current_question = data['suggested_follow_up']
                    print(f"🎤 Interviewer: {self.current_question}\n")
            else:
                print(f"❌ Error: {resp.status_code}")
        except Exception as e:
            print(f"❌ Error: {e}")
    
    def stats(self):
        """Show interview stats"""
        print(f"\n📊 Interview Stats:")
        print(f"   Turns: {self.turn}")
        if self.scores:
            avg = sum(self.scores) / len(self.scores)
            print(f"   Average Score: {avg:.1f}/10")
            print(f"   Scores: {self.scores}")

# Start the interview
interview = InterviewChat(api_url, job_role, company_context)
interview.start()

In [ ]:
# Answer the first question
answer = "I have 3 years of experience with Python and Django. I've built several REST APIs for e-commerce platforms."
interview.answer(answer)

In [ ]:
# Continue answering more questions
answer2 = "One project I'm proud of involved optimizing a slow API. I identified N+1 queries using Django Debug Toolbar, then used select_related() to fetch related objects. Reduced query count from 50+ to just 2."
interview.answer(answer2)

In [ ]:
# View final stats
interview.stats()

---

## 📝 Tips for Best Results

1. **Use Streamlit** (Option A) - Better user experience with web interface
2. **Use Direct Python** (Option B) - Run directly in cells, no extra setup
3. **Be specific** - Share real project examples and metrics
4. **Natural language** - Answer like you're talking to a real person
5. **Ask clarifications** - It's okay to ask what the interviewer wants to know

---

## 🔗 Resources

- **API Docs**: {public_url}/docs
- **GitHub**: https://github.com/yourusername/AI-Interview-Agent
- **Documentation**: Check README.md for full details